# ukpyn for Local Authority Energy Strategy

This notebook demonstrates how **ukpyn** provides access to the UK Power Networks 
data that underpins local authority energy strategy work.

**What you'll learn:**

1. How ukpyn simplifies access to 133+ ODP datasets
2. Demand, capacity and headroom data for planning
3. Embedded generation and low-carbon technology deployment
4. Geospatial data — substations, lines, and areas on a map
5. Connection pipeline and network constraints
6. Answering real strategy questions with data

**Prerequisites:**
- `pip install "ukpyn[all]"` (includes pandas and plotting dependencies)
- A UK Power Networks ODP API key set as `UKPN_API_KEY` in your environment or `.env` file
- Complete [01-getting-started.ipynb](01-getting-started.ipynb) if you are new to ukpyn

## Setup

Import the orchestrators we'll use. Each orchestrator groups related datasets behind simple function calls — no need to memorise dataset IDs.

In [20]:
import pandas as pd

from ukpyn import ders, dfes, dnoa, gis, ltds
from ukpyn.registry import ALL_DATASETS

# Quick check — how many datasets does ukpyn know about?
print(f"ukpyn knows {len(ALL_DATASETS)} dataset aliases")

ukpyn knows 118 dataset aliases


---

## 1. Demand & Capacity — the foundation of any energy strategy

Before recommending heat pumps, EV chargers, or solar, you need to know:
- What is the **current peak demand** at each substation?
- What is the **forecast demand** under different scenarios?
- How much **headroom** (spare capacity) is available?

### 1a. Observed peak demand (LTDS Table 3a)

Table 3a gives the measured peak demand at every primary substation, published twice yearly.

In [42]:
# What methods does the ltds orchestrator provide?

print(repr(ltds))

LTDSOrchestrator(methods=[export, get, get_async, get_cim, get_projects, get_table_1, get_table_2a, get_table_2b, get_table_3a, get_table_3b, get_table_4a, get_table_4b, get_table_5, get_table_6, get_table_7, get_table_8])


In [22]:
# Get observed peak demand for Eastern Power Networks
observed = ltds.get_table_3a(licence_area="EPN", limit=10)
print(f"Total records: {observed.total_count}")
print(f"Returned: {len(observed.records)}")
print()

# Show the first record's fields
rec = observed.records[0]
for key, val in sorted(rec.fields.items()):
    print(f"  {key}: {val}")

Total records: 1110
Returned: 10

  firm_capacity_mw: 9.2
  forecast_m_d_mw_25_26: 7.0
  forecast_m_d_mw_26_27: 7.1
  forecast_m_d_mw_27_28: 7.3
  forecast_m_d_mw_28_29: 7.5
  forecast_m_d_mw_29_30: 7.8
  functional_location: EPN-S0000000H8158
  gridsupplypoint: Amersham
  licencearea: Eastern Power Networks (EPN)
  maximum_demand_24_25_mw: 7.0
  maximum_demand_24_25_pf: 0.92
  minimum_load_scaling_factor: 25.2%
  season: Summer
  substation: Ilmer Primary 11kV
  unutilised_capacity_percent: 23.913043478260864


### 1b. Forecast demand (LTDS Table 3b)

Table 3b provides the true (firm) demand forecasts. Compare this with Table 3a to see the gap between observed and forecast.

In [23]:
forecast = ltds.get_table_3b(licence_area="EPN", limit=5)

print(f"Forecast demand records (EPN): {forecast.total_count}")
for rec in forecast.records[:3]:
    site = rec.fields.get("sitefunctionallocation", "unknown")
    print(f"  {site}: {rec.fields}")

Forecast demand records (EPN): 1110
  unknown: {'gridsupplypoint': 'Amersham', 'substation': 'Coldharbour Farm 11kV', 'season': 'Winter', 'maximum_demand_24_25_mw': 4.2, 'maximum_demand_24_25_pf': 0.96, 'forecast_m_d_mw_25_26': 4.2, 'forecast_m_d_mw_26_27': 4.1, 'forecast_m_d_mw_27_28': 4.1, 'forecast_m_d_mw_28_29': 4.0, 'forecast_m_d_mw_29_30': 4.0, 'firm_capacity_mw': 14.4, 'minimum_load_scaling_factor': None, 'functional_location': 'EPN-S0000000H8009', 'licencearea': 'Eastern Power Networks (EPN)'}
  unknown: {'gridsupplypoint': 'Amersham', 'substation': 'Coldharbour Farm 11kV', 'season': 'Summer', 'maximum_demand_24_25_mw': 2.9, 'maximum_demand_24_25_pf': 0.96, 'forecast_m_d_mw_25_26': 2.9, 'forecast_m_d_mw_26_27': 2.9, 'forecast_m_d_mw_27_28': 2.9, 'forecast_m_d_mw_28_29': 2.8, 'forecast_m_d_mw_29_30': 2.8, 'firm_capacity_mw': 11.5, 'minimum_load_scaling_factor': '28.8%', 'functional_location': 'EPN-S0000000H8009', 'licencearea': 'Eastern Power Networks (EPN)'}
  unknown: {'gridsu

### 1c. Network headroom by scenario (DFES)

The DFES headroom report shows capacity availability at each substation under different future energy scenarios (e.g. Net Zero, Leading the Way). This is the key dataset for answering *"can the network support X?"*.

In [24]:
headroom = dfes.get_headroom(limit=10)

print(f"Headroom records: {headroom.total_count}")
print()

# Show scenario, substation, year, and headroom value
for rec in headroom.records[:5]:
    f = rec.fields
    print(
        f"  {f.get('scenario', '?'):30s} | "
        f"{f.get('sitefunctionallocation', '?'):25s} | "
        f"Year: {f.get('year', '?')} | "
        f"Headroom: {f.get('headroom_mw', '?')} MW"
    )

Headroom records: 176580

  Electric Engagement            | LPN-S000000000523         | Year: 2024 | Headroom: 55.3 MW
  Electric Engagement            | LPN-S000000000523         | Year: 2025 | Headroom: 55.1 MW
  Electric Engagement            | LPN-S000000000523         | Year: 2031 | Headroom: 54.1 MW
  Electric Engagement            | LPN-S000000000523         | Year: 2032 | Headroom: 53.9 MW
  Electric Engagement            | LPN-S000000000523         | Year: 2033 | Headroom: 53.8 MW


### 1d. DFES scenarios by local authority

This dataset has one row per local authority × pathway × year, with individual columns for each technology type (domestic heat pumps, EVs, rooftop solar, batteries, etc.) — directly aligned with how strategy documents are structured.

In [43]:
la_scenarios = dfes.get(
    "by_local_authority",
    limit=10,
    where="lad22nm = 'Mole Valley'",
)

print(f"LA-level DFES records for Mole Valley: {la_scenarios.total_count}")
print()

# Each row is one pathway + year with individual LCT columns
for rec in la_scenarios.records[:5]:
    f = rec.fields
    la_name = f.get("lad22nm", "?")
    pathway = f.get("pathway", "?")
    year = f.get("year", "?")
    heat_pumps = f.get("number_of_domestic_heat_pumps", "?")
    evs = f.get("number_of_electric_cars_hybrid_and_full_electric", "?")
    solar_kw = f.get(
        "sum_of_small_rooftop_solar_generation_less_than_or_equal_to_4_kw", "?"
    )
    print(
        f"  {la_name:25s} | {pathway:20s} | {year} | "
        f"Heat pumps: {heat_pumps:>8} | EVs: {evs:>8} | Solar ≤4kW: {solar_kw:>8}"
    )

LA-level DFES records for Mole Valley: 6048

  Mole Valley               | Holistic Transition  | 2032 | Heat pumps:       14 | EVs:      545 | Solar ≤4kW:      469
  Mole Valley               | Holistic Transition  | 2033 | Heat pumps:       15 | EVs:      611 | Solar ≤4kW:      477
  Mole Valley               | Holistic Transition  | 2034 | Heat pumps:       17 | EVs:      673 | Solar ≤4kW:      487
  Mole Valley               | Holistic Transition  | 2039 | Heat pumps:      115 | EVs:      908 | Solar ≤4kW:      549
  Mole Valley               | Holistic Transition  | 2042 | Heat pumps:      170 | EVs:      994 | Solar ≤4kW:      601


---

## 2. What's already connected — embedded generation & LCT

To plan what *should* go where, you first need to know what's *already* there.

### 2a. Embedded Capacity Register (ECR) — all DER ≥ 1MW

In [26]:
# Filter to a specific local authority — use a where clause
ecr = ders.get(
    "embedded_capacity",
    limit=10,
    where="local_authority = 'Mole Valley'",
)

print(f"Embedded capacity sites in Mole Valley (≥1MW): {ecr.total_count}")
print()

for rec in ecr.records[:5]:
    f = rec.fields
    source = f.get("energy_source_1") or "?"
    capacity = f.get("maximum_export_capacity_mw") or "?"
    status = f.get("connection_status") or "?"
    postcode = f.get("postcode") or "?"
    print(f"  {source:20s} | {str(capacity):>8} MW | {status:15s} | {postcode}")
    if rec.geometry:
        print(f"    ↳ geometry: {rec.geometry}")

Embedded capacity sites in Mole Valley (≥1MW): 6

  Solar                |      3.4 MW | Connected       | KT22 9DA
  Solar                |        ? MW | Accepted to Connect | KT11 3QH
  Fossil - Gas         |      1.1 MW | Connected       | RH5 5JL
  Stored Energy (all stored energy irrespective of the original energy source) |        ? MW | Accepted to Connect | KT21 2BU
  Stored Energy (all stored energy irrespective of the original energy source) |      6.0 MW | Connected       | RH4 3LY


### 2b. Low-carbon technology deployment by LSOA

Heat pump, EV, and solar density at Lower Super Output Area level — ideal for identifying where uptake is high or low relative to network capacity.

In [27]:
from ukpyn.registry import SMART_LCT_DATASETS

# Show all LCT-related datasets available
print("LCT & Smart Meter datasets:")
for name, dataset_id in SMART_LCT_DATASETS.items():
    print(f"  {name:30s} → {dataset_id}")

LCT & Smart Meter datasets:
  smart_meter_substation         → ukpn-smart-meter-consumption-substation
  smart_meter_feeder             → ukpn-smart-meter-consumption-lv-feeder
  smart_meter_volumes            → ukpn-smart-meter-installation-volumes
  lct_lsoa                       → ukpn-low-carbon-technologies-lsoa
  lct_secondary                  → ukpn-low-carbon-technologies-secondary
  low_carbon_tech                → low-carbon-technologies


In [28]:
from ukpyn import UKPNClient


async def fetch_lct_sample():
    async with UKPNClient() as client:
        return await client.get_records("ukpn-low-carbon-technologies-lsoa", limit=5)


lct = await fetch_lct_sample()
print(f"LCT LSOA records: {lct.total_count}")
for rec in lct.records[:3]:
    f = rec.fields
    print(
        f"  LSOA: {f.get('lsoa21cd', '?')} | "
        f"LA: {f.get('local_authority', '?')} | "
        f"Type: {f.get('type', '?')} | "
        f"Connections: {f.get('lct_connections', '?')} | "
        f"Export: {f.get('exportrating', '?')} kW"
    )

LCT LSOA records: 36044
  LSOA: E01024717 | LA: Tonbridge and Malling | Type: EV Charging Point | Connections: 22.0 | Export: 0.0 kW
  LSOA: E01024722 | LA: Tonbridge and Malling | Type: EV Charging Point | Connections: 16.0 | Export: 0.0 kW
  LSOA: E01024725 | LA: Tonbridge and Malling | Type: EV Charging Point | Connections: 21.0 | Export: 0.0 kW


---

## 3. Where things are — geospatial data

The GIS orchestrator provides access to substation locations, overhead line routes, poles, and boundary polygons. Every record has a normalised `.geometry` property — standard GeoJSON, ready for any mapping tool.

### 3a. Primary substations with locations

In [29]:
substations = gis.get_primary_substations(licence_area="SPN", limit=10)

print(f"Primary substations (SPN): {substations.total_count}")
print()

for rec in substations.records[:5]:
    f = rec.fields
    name = f.get("sitefunctionallocation", "?")
    voltage = f.get("sitevoltage", "?")
    geom = rec.geometry
    print(f"  {str(name):30s} | {str(voltage):>6} kV | geometry: {geom}")

Primary substations (SPN): 270

  SPN-S000000008020              |     33 kV | geometry: {'type': 'Point', 'coordinates': [1.126801, 51.088393]}
  SPN-S000000008025              |     33 kV | geometry: {'type': 'Point', 'coordinates': [-0.362501, 51.422094]}
  SPN-S000000008129              |     33 kV | geometry: {'type': 'Point', 'coordinates': [0.434154, 50.84336]}
  SPN-S000000008194              |     33 kV | geometry: {'type': 'Point', 'coordinates': [0.830319, 51.402899]}
  SPN-S000000008231              |     33 kV | geometry: {'type': 'Point', 'coordinates': [0.243288, 50.858428]}


### 3b. Secondary sites — fine-grained network coverage

There are ~60,000+ secondary substations. Each record includes customer count, headroom, and utilisation.

In [30]:
secondary = gis.get_secondary_sites(limit=5)

print(f"Secondary sites total: {secondary.total_count}")
print()

for rec in secondary.records[:3]:
    f = rec.fields
    print(
        f"  {str(f.get('functionallocation', '?')):30s} | "
        f"Customers: {str(f.get('customer_count', '?')):>5} | "
        f"Headroom: {str(f.get('demand_headroom', '?')):>6} | "
        f"Geometry: {rec.geometry}"
    )

Secondary sites total: 72750

  SPN-S000000312064              | Customers:   376 | Headroom: 20-40% | Geometry: {'type': 'Point', 'coordinates': [0.603182, 51.360074]}
  SPN-S000000392928              | Customers:     8 | Headroom: 40-60% | Geometry: {'type': 'Point', 'coordinates': [0.489418, 51.177493]}
  EPN-S0000008U7949              | Customers:   340 | Headroom: 20-40% | Geometry: {'type': 'Point', 'coordinates': [-0.417579, 51.565478]}


### 3C. GeoJSON export — ready for QGIS or web maps

Export an entire dataset as GeoJSON bytes. Use the `dimensions` parameter to control 2D vs 3D output:
- `"2d"` — strips Z elevation values (best for flat web maps like Leaflet/Mapbox)
- `"3d"` — ensures all coordinates have Z (for 3D viewers)
- `"raw"` — pass through as-is (default)

In [32]:
import json

# Export licence boundaries as 2D GeoJSON
geojson_bytes = gis.export_geojson("licence_boundaries", dimensions="2d")
geojson = json.loads(geojson_bytes)

print(f"GeoJSON type: {geojson['type']}")
print(f"Features: {len(geojson['features'])}")

# Peek at the first feature
feat = geojson["features"][0]
props = feat.get("properties", {})
geom_type = feat["geometry"]["type"]
print(f"\nFirst feature: {props.get('name', props)} — {geom_type}")

# Verify coordinates are 2D (no Z value)
sample_coord = feat["geometry"]["coordinates"]
# Navigate to a leaf coordinate
while isinstance(sample_coord[0], list):
    sample_coord = sample_coord[0]
print(f"Sample coordinate (should be [lon, lat]): {sample_coord}")

GeoJSON type: FeatureCollection
Features: 3

First feature: EPN — Polygon
Sample coordinate (should be [lon, lat]): [-0.656437679317974, 52.0053970419287]


---

## 4. Connection pipeline & network constraints

Understanding what's in the connection queue and where the network is constrained helps prioritise strategy recommendations.

### 4a. Connection interest queue (LTDS Table 6)

In [33]:
connections = ltds.get("table_6", limit=10)

print(f"Connection interest records: {connections.total_count}")
print()

for rec in connections.records[:5]:
    f = rec.fields
    substation = f.get("substation", "?")
    demand_cap = f.get("demand_numbers_received_total_capacity", "?")
    gen_cap = f.get("generation_numbers_received_total_capacity", "?")
    status = f.get("status_of_connection", "?")
    print(
        f"  {str(substation):35s} | "
        f"Demand: {str(demand_cap):>8} MW | "
        f"Gen: {str(gen_cap):>8} MW | "
        f"Status: {status}"
    )

Connection interest records: 803

  Romford Primary 11kV                | Demand:      4.4 MW | Gen:     None MW | Status: Connection offers made (not yet accepted by customer)
  Chequers Primary 11kV               | Demand:      5.1 MW | Gen:     None MW | Status: Connection offers made (not yet accepted by customer)
  Chelmsford North Grid 11kV          | Demand:     12.1 MW | Gen:     None MW | Status: Connection offers made (not yet accepted by customer)
  Chelmsford North Grid 11kV          | Demand:     None MW | Gen:        8 MW | Status: Budget Estimates Provided
  Lake & Elliot Primary 11kV          | Demand:      4.9 MW | Gen:     None MW | Status: Connection offers made (cancelled, expired, superseded)


### 4b. Infrastructure projects — planned reinforcements

The 5-year infrastructure plan shows where UKPN is investing. Critical for knowing if constraints will be relieved.

In [34]:
projects = ltds.get("infrastructure_projects", limit=10)

print(f"Infrastructure project records: {projects.total_count}")
print()

for rec in projects.records[:5]:
    f = rec.fields
    name = f.get("substation_or_circuit_ple_name", "?")
    asset = f.get("asset_type_or_quantity", "?")
    start = f.get("expected_start_year", "?")
    end = f.get("expected_completion_year", "?")
    justification = f.get("justification_for_the_need", "?")
    print(f"  {str(name):35s} | {str(asset):25s} | {start}–{end} | {justification}")
    if rec.geometry:
        print(f"    ↳ location: {rec.geometry}")

Infrastructure project records: 41

  Burwell Local Grid 33               | 1 x 33kV CB, 2 x grid transformers uprated | 2023–2026 | Load related reinforcement
    ↳ location: {'type': 'Point', 'coordinates': [0.31728, 52.277505]}
  East Cambridge 33kV                 | Establish new Grid substation | 2023–2027 | Load related reinforcement
    ↳ location: {'type': 'Point', 'coordinates': [0.14808796, 52.170269]}
  Fulbourn/Sawston 33kV               | Additional 33kV Circuit   | 2023–2026 | Load related reinforcement
    ↳ location: {'type': 'Point', 'coordinates': [0.187199, 52.188599]}
  Radnor                              | Installation of a third 33kV circuit and third primary transformer | 2023–2026 | Load related reinforcement
    ↳ location: {'type': 'Point', 'coordinates': [0.141732, 52.18929]}
  Brockenhurst Primary 11kV, Mill Hill Primary 11kV | Asset replacement. Increase in Transformer Capacity and 11kV Network Reinforcement | 2017–2026 | Asset replacement
    ↳ location: {

### 4c. DNOA — flex vs reinforcement decisions

The Distribution Network Options Assessment shows where UKPN has chosen flexibility services over traditional reinforcement — relevant for understanding which areas may benefit from demand-side solutions.

In [35]:
assessment = dnoa.get_assessment(limit=10)

print(f"DNOA records: {assessment.total_count}")
print()

for rec in assessment.records[:5]:
    f = rec.fields
    print(
        f"  {f.get('substation_title', '?'):30s} | "
        f"Result: {f.get('dnoa_result', '?'):20s} | "
        f"Area: {f.get('area', '?')}"
    )

DNOA records: 19

  Chatteris                      | Result: Flexibility          | Area: EPN
  Coxford                        | Result: Flexibility          | Area: EPN
  Hendon Way                     | Result: Flexibility          | Area: EPN
  St Anthony Street              | Result: Flexibility          | Area: EPN
  Stody                          | Result: Flexibility          | Area: EPN


---

## 5. Answering real strategy questions

Here's how to combine these datasets to answer the kinds of questions local authorities actually ask.

### Q: "Is there capacity for more heat pumps in our area?"

Combine headroom data with current LCT deployment to find substations with spare capacity.

In [36]:
# Fetch headroom data
headroom_data = dfes.get_headroom(limit=50)

# Convert to a DataFrame for easier filtering
rows = [rec.fields for rec in headroom_data.records]
df = pd.DataFrame(rows)

print(f"Columns: {list(df.columns)}")
print(f"Rows: {len(df)}")

# Show substations with positive headroom (spare capacity)
if "headroom_mw" in df.columns:
    df["headroom_mw"] = pd.to_numeric(df["headroom_mw"], errors="coerce")
    has_capacity = df[df["headroom_mw"] > 0]
    print(f"\nSubstations with positive headroom: {len(has_capacity)} of {len(df)}")
    print(
        has_capacity[["sitefunctionallocation", "headroom_mw", "scenario"]]
        .head(10)
        .to_string(index=False)
    )

Columns: ['category', 'licencearea', 'substation_name', 'voltage_kv', 'sitefunctionallocation', 'bulksupplypoint', 'gridsupplypoint', 'scenario', 'year', 'headroom_mw', 'spatial_coordinates']
Rows: 50

Substations with positive headroom: 50 of 50
sitefunctionallocation  headroom_mw            scenario
     LPN-S000000000523         55.3 Electric Engagement
     LPN-S000000000523         55.1 Electric Engagement
     LPN-S000000000523         54.1 Electric Engagement
     LPN-S000000000523         53.9 Electric Engagement
     LPN-S000000000523         53.8 Electric Engagement
     LPN-S000000000523         53.4 Electric Engagement
     LPN-S000000000523         52.3 Electric Engagement
     LPN-S000000000523         50.8 Electric Engagement
     LPN-S000000000796         31.2 Electric Engagement
     LPN-S000000000796         50.9 Electric Engagement


### Q: "What generation is already connected near a specific substation?"

In [37]:
# Fetch generation data from LTDS Table 5
generation = ltds.get("table_5", limit=20)

gen_rows = [rec.fields for rec in generation.records]
gen_df = pd.DataFrame(gen_rows)

print(f"Generation records: {generation.total_count}")
print(f"Columns: {list(gen_df.columns)}")
gen_df.head()

Generation records: 769
Columns: ['gridsupplypoint', 'substation', 'connection_voltage_kv', 'installedcapacity_mva', 'fuel_type', 'connected_accepted', 'spatial_coordinates', 'sitefunctionallocation', 'licencearea']


,gridsupplypoint,substation,connection_voltage_kv,installedcapacity_mva,fuel_type,connected_accepted,spatial_coordinates,sitefunctionallocation,licencearea
0,Amersham,None,132,210.5,Energy Storage (>=1MW),Accepted,None,None,Eastern Power Networks (EPN)
1,Amersham,None,132,104.2,Energy Storage (>=1MW),Connected,None,None,Eastern Power Networks (EPN)
2,BIGGLESWADE 132KV,None,132,157.7,Photovoltaic (>=1MW),Accepted,None,None,Eastern Power Networks (EPN)
3,Bramford,None,132,147.4,Energy Storage (>=1MW),Connected,None,None,Eastern Power Networks (EPN)
4,Brimsdown (EPN),None,132,157.8,Photovoltaic (>=1MW),Accepted,None,None,Eastern Power Networks (EPN)


---

## 6. Data quality — what ukpyn handles for you

The ODP has some data quality quirks that ukpyn normalises automatically:

| Issue | What happens in the raw API | What ukpyn does |
|---|---|---|
| **NaN values** | Fields like `grid_site` return `float('nan')` — invalid JSON | Replaced with `None` on every record |
| **Feature wrappers** | Some `geo_shape` fields return `{"type": "Feature", "geometry": {...}}` | Unwrapped to bare geometry automatically |
| **Multiple geometry formats** | `geo_shape`, `spatial_coordinates`, `geo_point_2d`, `geopoint` | Unified via `record.geometry` property |
| **Mixed 2D/3D coordinates** | Exports have Z values, records don't | `export_geojson(dimensions="2d")` to normalise |

This means you can serialise records to JSON, write to PostGIS, or feed into a web map without worrying about edge cases.

In [38]:
import json

# Demonstrate that records serialise cleanly to JSON
rec = substations.records[0]

# This would fail with raw ODP data if NaN values were present
json_str = json.dumps(rec.fields)
print("JSON serialisation: OK")
print(f"Geometry type: {type(rec.geometry)}")
print(f"Geometry value: {rec.geometry}")

JSON serialisation: OK
Geometry type: <class 'dict'>
Geometry value: {'type': 'Point', 'coordinates': [1.126801, 51.088393]}


---

## 7. Available datasets — full reference

Here's everything ukpyn provides access to, grouped by domain.

In [39]:
from ukpyn.registry import (
    CONNECTIONS_DATASETS,
    CURTAILMENT_DATASETS,
    DER_DATASETS,
    DFES_DATASETS,
    DNOA_DATASETS,
    EQUIPMENT_DATASETS,
    FLEXIBILITY_DATASETS,
    GEO_DATASETS,
    LTDS_DATASETS,
    POWERFLOW_DATASETS,
    REFERENCE_DATASETS,
    SMART_LCT_DATASETS,
)

groups = {
    "LTDS (network planning)": LTDS_DATASETS,
    "DFES (future scenarios)": DFES_DATASETS,
    "DNOA (flex vs reinforcement)": DNOA_DATASETS,
    "GIS (geospatial assets)": GEO_DATASETS,
    "DER (embedded generation)": DER_DATASETS,
    "Flexibility & Curtailment": {**FLEXIBILITY_DATASETS, **CURTAILMENT_DATASETS},
    "Powerflow (time series)": POWERFLOW_DATASETS,
    "Smart Meters & LCT": SMART_LCT_DATASETS,
    "Connections & Queue": CONNECTIONS_DATASETS,
    "Equipment": EQUIPMENT_DATASETS,
    "Reference": REFERENCE_DATASETS,
}

for group_name, datasets in groups.items():
    # Deduplicate aliases (multiple friendly names → same ID)
    unique_ids = set(datasets.values())
    print(f"\n{group_name} ({len(unique_ids)} datasets):")
    for name, did in datasets.items():
        print(f"  {name:35s} → {did}")


LTDS (network planning) (14 datasets):
  table_1                             → ltds-table-1-circuit-data
  circuit_data                        → ltds-table-1-circuit-data
  table_2a                            → ltds-table-2a-transformer-2w
  transformer_2w                      → ltds-table-2a-transformer-2w
  table_2b                            → ltds-table-2b-transformer-data-3w
  transformer_3w                      → ltds-table-2b-transformer-data-3w
  table_3a                            → ltds-table-3a-load-data-observed
  observed_demand                     → ltds-table-3a-load-data-observed
  observed_peak_demand                → ltds-table-3a-load-data-observed
  table_3a_transposed                 → ltds-table-3a-load-data-observed-transposed
  table_3b                            → ltds-table-3b-load-data-true
  forecast_demand                     → ltds-table-3b-load-data-true
  table_4a                            → ltds-table-4a-3ph-fault-level
  fault_level_3ph              

---

## Summary

| Strategy question | Key datasets | Orchestrator |
|---|---|---|
| Is there capacity for heat pumps / EVs? | `headroom`, `lct_lsoa`, `lct_secondary` | `dfes` |
| What's the current peak demand? | `table_3a`, `table_3b` | `ltds` |
| What generation is already connected? | `ecr`, `ecr_small`, `table_5` | `ders`, `ltds` |
| Where are the substations? | `grid_primary_sites`, `secondary_sites` | `gis` |
| What reinforcement is planned? | `infrastructure_projects` | `ltds` |
| Which areas are constrained? | `dnoa`, `headroom` (negative values) | `dnoa`, `dfes` |
| Where should EV chargers go? | `chargepoints`, `headroom`, `by_local_authority` | `dfes` + client |
| What does the network look like on a map? | `hv_overhead_lines`, `licence_boundaries` + `export_geojson()` | `gis` |

For more details, see the [full documentation](https://ukpn-dso.github.io/ukpyn/) and the other [tutorials](README.md).